# Transaction Feature Engineering

See the repository README and data/README.md before running. Outputs were cleared for publication.

In [ ]:
from pathlib import Path
import os
ROOT = Path.cwd() if (Path.cwd() / "notebooks").is_dir() else Path.cwd().parent
DATA_DIR = Path(os.environ.get("PROJECT_DATA_DIR", str(ROOT / "data")))
ARTIFACT_DIR = ROOT / "artifacts"
ARTIFACT_DIR.mkdir(exist_ok=True)


In [ ]:
# Use scripts/prepare_customer_data.py for the local data workflow.

In [ ]:
import pandas as pd
import numpy as np
import json

PATH = str(DATA_DIR) + '/'

transactions = pd.read_csv(PATH + 'transactions_data.csv')
users        = pd.read_csv(PATH + 'users_data.csv')
cards        = pd.read_csv(PATH + 'cards_data.csv')

with open(PATH + 'mcc_codes.json') as f:
    mcc_raw = json.load(f)

with open(PATH + 'train_fraud_labels.json') as f:
    fraud_raw = json.load(f)

print("transactions:", transactions.shape)
print("users       :", users.shape)
print("cards       :", cards.shape)
print("mcc codes   :", len(mcc_raw))
print("fraud labels:", len(fraud_raw))

In [ ]:
# Fraud labels ka actual structure dekho
print(type(fraud_raw))
print(str(fraud_raw)[:500])

In [ ]:
# Fraud labels sahi se parse karo
fraud_labels = pd.DataFrame(
    list(fraud_raw['target'].items()),
    columns=['transaction_id', 'is_fraud']
)

# Yes/No → 1/0
fraud_labels['transaction_id'] = fraud_labels['transaction_id'].astype(int)
fraud_labels['is_fraud'] = fraud_labels['is_fraud'].map({'Yes': 1, 'No': 0})

print("Fraud labels shape:", fraud_labels.shape)
print("Fraud distribution:\n", fraud_labels['is_fraud'].value_counts())

In [ ]:
mcc_codes = pd.DataFrame(
    list(mcc_raw.items()),
    columns=['mcc', 'mcc_category']
)
mcc_codes['mcc'] = mcc_codes['mcc'].astype(int)

print("MCC codes shape:", mcc_codes.shape)
print(mcc_codes.head(3))

In [ ]:
transactions = transactions.rename(columns={'id': 'transaction_id'})
users        = users.rename(columns={'id': 'client_id'})
cards        = cards.rename(columns={'id': 'card_id'})

In [ ]:
transactions.shape

In [ ]:
transactions['amount'] = (
    transactions['amount']
    .astype(str)
    .str.replace('$', '', regex=False)
    .astype(float)
)

transactions['date'] = pd.to_datetime(transactions['date'])

transactions['mcc'] = pd.to_numeric(transactions['mcc'], errors='coerce')

In [ ]:
transactions = transactions.merge(mcc_codes, on='mcc', how='left')

print("Shape:", transactions.shape)
print(transactions[['mcc', 'mcc_category']].head(3))

In [ ]:
# Transactions + Users merge
master = transactions.merge(
    users,
    on='client_id',
    how='left'
)

print("Shape after users merge:", master.shape)

In [ ]:
card_cols = ['card_id', 'card_brand', 'card_type', 
             'credit_limit', 'has_chip', 'num_cards_issued', 
             'acct_open_date', 'card_on_dark_web']

master = master.merge(
    cards[card_cols],
    on='card_id',
    how='left'
)

print("Shape after cards merge:", master.shape)
print("Columns:", master.columns.tolist())

In [ ]:
# Master table save karo
master.to_csv(str(DATA_DIR / "master_transactions.csv"), index=False)
print("✓ Master table saved")

# Fraud labels bhi save karo
fraud_labels.to_csv(str(DATA_DIR / "fraud_labels.csv"), index=False)
print("✓ Fraud labels saved")

In [ ]:
import pandas as pd
import numpy as np

master       = pd.read_csv(str(DATA_DIR / "master_transactions.csv"))
fraud_labels = pd.read_csv(str(DATA_DIR / "fraud_labels.csv"))

print("master      :", master.shape)
print("fraud_labels:", fraud_labels.shape)

In [ ]:
print("Shape:", master.shape)
print("\nData types:")
print(master.dtypes)
print("\nNull values:")
print(master.isnull().sum()[master.isnull().sum() > 0])

In [ ]:
dollar_cols = ['per_capita_income', 'yearly_income', 'total_debt', 'credit_limit']

for col in dollar_cols:
    master[col] = (master[col]
                   .astype(str)
                   .str.replace('$', '', regex=False)
                   .str.replace(',', '', regex=False)
                   .astype(float))

# Date fix
master['date'] = pd.to_datetime(master['date'])

print("Fixed dtypes:")
print(master[dollar_cols + ['date']].dtypes)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(master['amount'], bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('Spend Distribution (all)')
axes[0].set_xlabel('Amount ($)')
axes[0].set_ylabel('Count')

q99 = master['amount'].quantile(0.99)
filtered = master[master['amount'].between(0, q99)]
axes[1].hist(filtered['amount'], bins=50, color='steelblue', edgecolor='white')
axes[1].set_title('Spend Distribution (0 to 99th percentile)')
axes[1].set_xlabel('Amount ($)')

plt.tight_layout()
plt.show()

print("Min   :", master['amount'].min())
print("Max   :", master['amount'].max())
print("Mean  :", round(master['amount'].mean(), 2))
print("Median:", round(master['amount'].median(), 2))

In [ ]:
top_categories = (master['mcc_category']
                  .value_counts()
                  .head(10))

plt.figure(figsize=(12, 5))
top_categories.plot(kind='barh', color='steelblue', edgecolor='white')
plt.title('Top 10 Merchant Categories by Transaction Count')
plt.xlabel('Number of Transactions')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print(top_categories)

In [ ]:
txn_types = master['use_chip'].value_counts()

plt.figure(figsize=(8, 5))
txn_types.plot(kind='bar', color=['steelblue', 'coral', 'green'], edgecolor='white')
plt.title('Transaction Type Breakdown')
plt.xlabel('Transaction Type')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

print(txn_types)
print("\nPercentage:")
print(round(txn_types / txn_types.sum() * 100, 2))

In [ ]:
master['transaction_id'] = master['transaction_id'].astype(int)
fraud_check = master.merge(fraud_labels, on='transaction_id', how='left')
fraud_check['is_fraud'] = fraud_check['is_fraud'].fillna(0).astype(int)

fraud_counts = fraud_check['is_fraud'].value_counts()
labels = ['Normal', 'Fraud']

plt.figure(figsize=(6, 5))
plt.pie(fraud_counts, labels=labels, autopct='%1.2f%%',
        colors=['steelblue', 'coral'], startangle=90)
plt.title('Fraud vs Normal Transactions')
plt.tight_layout()
plt.show()

print(fraud_counts)
print(f"\nFraud rate: {round(fraud_counts[1]/fraud_counts.sum()*100, 3)}%")

In [ ]:
# Updated master with fraud labels save karo
fraud_check.to_csv(str(DATA_DIR / "master_with_fraud.csv"), index=False)
print("✓ Saved")

In [ ]:
master       = pd.read_csv(str(DATA_DIR / "master_transactions.csv"))

In [ ]:
master['date'] = pd.to_datetime( master['date'])
master['hour'] = master['date'].dt.hour
master['is_weekend'] = master['date'].dt.dayofweek.isin([5, 6]).astype(int)
master['is_night'] = (master['hour'].isin(range(22, 24 )) | master['hour'].isin(range(0 , 6 ))).astype(int)

print( master[['date' , 'hour' , 'is_weekend' , 'is_night']].head())

In [ ]:
master['has_error'] = master['errors'].notnull().astype(int)
print("Has error distribution:")
print(master['has_error'].value_counts())
print(f"\nError rate: {round(master['has_error'].mean()*100, 2)}%")

In [ ]:
master['is_online'] = (master['use_chip'] == 'Online Transaction').astype(int)
master['is_swipe']  = (master['use_chip'] == 'Swipe Transaction').astype(int)

print(master[['use_chip', 'is_online', 'is_swipe']].head(5))

In [ ]:
master['yearly_income'] = (master['yearly_income']
                           .astype(str)
                           .str.replace('$', '', regex=False)
                           .str.replace(',', '', regex=False)
                           .astype(float))

print(master['yearly_income'].head(5))
print("dtype:", master['yearly_income'].dtype)
master['total_debt'] = (master['total_debt']
                           .astype(str)
                           .str.replace('$', '', regex=False)
                           .str.replace(',', '', regex=False)
                           .astype(float))
print( master['total_debt'].head(5))
print( master['total_debt'].dtype)

In [ ]:
customer_features = master.groupby('client_id').agg(
    avg_transaction_amount = ('amount', 'mean'),
    total_spend            = ('amount', 'sum'),
    transaction_count      = ('amount', 'count'),
    std_transaction_amount = ('amount', 'std'),
    max_transaction_amount = ('amount', 'max'),
    online_txn_ratio       = ('is_online', 'mean'),
    swipe_txn_ratio        = ('is_swipe', 'mean'),
    night_txn_ratio        = ('is_night', 'mean'),
    weekend_txn_ratio      = ('is_weekend', 'mean'),
    merchant_diversity     = ('mcc_category', 'nunique'),
    error_rate             = ('has_error', 'mean'),
    yearly_income          = ('yearly_income', 'first'),
    credit_score = ('credit_score', 'first'),
    total_debt   = ('total_debt', 'first')
).reset_index()

print("Shape:", customer_features.shape)
customer_features.head(3)

In [ ]:
master['yearly_income'].dtype

In [ ]:
customer_features['spend_to_income_ratio'] = (
    customer_features['total_spend'] / customer_features['yearly_income']
)

print("Shape:", customer_features.shape)
customer_features.head(3)

In [ ]:
print(customer_features.isnull().sum())


In [ ]:
customer_features.to_csv(str(DATA_DIR / "customer_features.csv"), index=False)
print("✓ Saved")
print("\nFinal shape:", customer_features.shape)
print("\nFeatures:")
print(customer_features.columns.tolist())

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
customer_features = pd.read_csv(str(DATA_DIR / "customer_features.csv"))
customer_features.head()

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

customer_ids = customer_features['client_id']

features = customer_features.drop('client_id', axis=1)
scaler = StandardScaler()
features_scaled = scaler.fit_transform(features)

print("Scaled shape:", features_scaled.shape)

In [ ]:
pca_temp = PCA()
pca_temp.fit(features_scaled)

# Explained variance plot
plt.figure(figsize=(10, 5))
plt.plot(range(1, len(pca_temp.explained_variance_ratio_)+1),
         pca_temp.explained_variance_ratio_.cumsum(),
         marker='o', color='steelblue')
plt.axhline(y=0.90, color='red', linestyle='--', label='90% variance')
plt.axhline(y=0.95, color='green', linestyle='--', label='95% variance')
plt.xlabel('Number of Components')
plt.ylabel('Cumulative Explained Variance')
plt.title('PCA — How many components to keep?')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
cumsum = pca_temp.explained_variance_ratio_.cumsum()
for i, val in enumerate(cumsum):
    if val >= 0.90:
        print(f"90% variance at {i+1} components")
        break
for i, val in enumerate(cumsum):
    if val >= 0.95:
        print(f"95% variance at {i+1} components")
        break

In [ ]:
pca = PCA(n_components=min(10, *features_scaled.shape))
pca_result = pca.fit_transform(features_scaled)

print("PCA output shape:", pca_result.shape)
print(f"Total variance explained: {pca.explained_variance_ratio_.sum()*100:.2f}%")

In [ ]:
pca_df = pd.DataFrame(pca_result, columns=[f'PC{i+1}' for i in range(10)])
pca_df['client_id'] = customer_ids.values

print("PCA DataFrame shape:", pca_df.shape)
pca_df.head(3)

In [ ]:
# Dependencies are installed using requirements.txt.

import umap
reducer = umap.UMAP(n_components=2, random_state=42)
umap_result = reducer.fit_transform(pca_result)

print("UMAP output shape:", umap_result.shape)

In [ ]:
plt.figure(figsize=(10, 7))
plt.scatter(umap_result[:, 0], umap_result[:, 1], 
            alpha=0.6, s=15, color='steelblue')
plt.title('UMAP — Customer Spending Behavior (2D)')
plt.xlabel('UMAP 1')
plt.ylabel('UMAP 2')
plt.tight_layout()
plt.show()


In [ ]:
# Dependencies are installed using requirements.txt.

In [ ]:
import hdbscan

clusterer = hdbscan.HDBSCAN(min_cluster_size=15, min_samples=5)
cluster_labels = clusterer.fit_predict(umap_result)

print("Unique clusters:", len(set(cluster_labels[cluster_labels != -1])))
print("Noise points (-1):", (cluster_labels == -1).sum())
print("Cluster distribution:")
print(pd.Series(cluster_labels).value_counts().sort_index())

In [ ]:
clusterer = hdbscan.HDBSCAN(min_cluster_size=50, min_samples=3)
cluster_labels = clusterer.fit_predict(umap_result)

print("Unique clusters:", len(set(cluster_labels[cluster_labels != -1])))
print("Noise points (-1):", (cluster_labels == -1).sum())
print("Cluster distribution:")
print(pd.Series(cluster_labels).value_counts().sort_index())

In [ ]:
clusterer = hdbscan.HDBSCAN(min_cluster_size=20, min_samples=3)
cluster_labels = clusterer.fit_predict(umap_result)

print("Unique clusters:", len(set(cluster_labels[cluster_labels != -1])))
print("Noise points (-1):", (cluster_labels == -1).sum())
print("Cluster distribution:")
print(pd.Series(cluster_labels).value_counts().sort_index())

In [ ]:
umap_df = pd.DataFrame(umap_result, columns=['UMAP1', 'UMAP2'])
umap_df['cluster'] = cluster_labels
umap_df['client_id'] = customer_ids.values

# Colored UMAP plot
plt.figure(figsize=(10, 7))
colors = ['red' if c == -1 else plt.cm.tab10(c/7) for c in cluster_labels]
scatter = plt.scatter(umap_result[:, 0], umap_result[:, 1],
                      c=cluster_labels, cmap='tab10',
                      alpha=0.7, s=15)
plt.colorbar(scatter, label='Cluster')
plt.title('UMAP — Customer Clusters (HDBSCAN)')
plt.xlabel('UMAP 1')
plt.ylabel('UMAP 2')
plt.tight_layout()
plt.show()

In [ ]:
# Cluster features merge karo
customer_features['cluster'] = cluster_labels

# Har cluster ka average dekho
cluster_profile = customer_features.groupby('cluster').agg({
    'avg_transaction_amount': 'mean',
    'total_spend'           : 'mean',
    'transaction_count'     : 'mean',
    'online_txn_ratio'      : 'mean',
    'swipe_txn_ratio'       : 'mean',
    'night_txn_ratio'       : 'mean',
    'weekend_txn_ratio'     : 'mean',
    'merchant_diversity'    : 'mean',
    'error_rate'            : 'mean',
    'spend_to_income_ratio' : 'mean',
}).round(3)

print(cluster_profile)

In [ ]:
cluster_names = {
    -1: 'Noise / Outliers',
     0: 'Moderate Spenders',        # avg spend ok, low night txn
     1: 'Debt-Stressed Spenders',   # spend_to_income 173 — bahut zyada!
     2: 'High Value Customers',     # highest avg amount + total spend
     3: 'Online Shoppers',          # online_txn_ratio 0.368 — sabse zyada
     4: 'Night Owls',               # night_txn_ratio 0.245 — raat ko spend
     5: 'Late Night Spenders',      # night_txn_ratio 0.399 — sabse zyada raat
     6: 'Frugal Spenders',          # lowest avg amount + lowest spend
}

# Names add karo
umap_df['cluster_name'] = umap_df['cluster'].map(cluster_names)
customer_features['cluster_name'] = customer_features['cluster'].map(cluster_names)

print("Cluster names assigned!")
print(customer_features['cluster_name'].value_counts())

In [ ]:
# Named UMAP plot
import matplotlib.patches as mpatches

colors_map = {
    'Noise / Outliers'      : 'grey',
    'Moderate Spenders'     : 'orange',
    'Debt-Stressed Spenders': 'green',
    'High Value Customers'  : 'brown',
    'Online Shoppers'       : 'blue',
    'Night Owls'            : 'purple',
    'Late Night Spenders'   : 'yellow',
    'Frugal Spenders'       : 'cyan',
}

plt.figure(figsize=(12, 8))
for name, color in colors_map.items():
    mask = umap_df['cluster_name'] == name
    plt.scatter(umap_df.loc[mask, 'UMAP1'],
                umap_df.loc[mask, 'UMAP2'],
                c=color, label=name, alpha=0.7, s=20)

plt.title('Customer Spending Personalities')
plt.xlabel('UMAP 1')
plt.ylabel('UMAP 2')
plt.legend(loc='lower right', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Phase 7 — Isolation Forest (Fraud Detection)
from sklearn.ensemble import IsolationForest

# Original scaled features pe run karo
iso_forest = IsolationForest(n_estimators=100, 
                              contamination=0.05, 
                              random_state=42)

anomaly_scores = iso_forest.fit_predict(features_scaled)
anomaly_score_values = iso_forest.decision_function(features_scaled)

# -1 = anomaly, 1 = normal
customer_features['anomaly'] = anomaly_scores
customer_features['anomaly_score'] = anomaly_score_values

print("Anomaly distribution:")
print(pd.Series(anomaly_scores).value_counts())
print(f"\nFlagged as fraud risk: {(anomaly_scores == -1).sum()} customers")

In [ ]:
# Fraud labels ke saath validate karo
# Transaction level pe fraud check karo per customer
fraud_per_customer = fraud_check.groupby('client_id')['is_fraud'].sum().reset_index()
fraud_per_customer.columns = ['client_id', 'fraud_txn_count']
fraud_per_customer['is_high_fraud'] = (fraud_per_customer['fraud_txn_count'] > 5).astype(int)
fraud_per_customer['has_fraud'] = (fraud_per_customer['fraud_txn_count'] > 0).astype(int)

# Customer features ke saath merge karo
validation = customer_features[['client_id', 'anomaly']].merge(
    fraud_per_customer, on='client_id', how='left'
)
validation['has_fraud'] = validation['has_fraud'].fillna(0).astype(int)

# Results
print("Validation Results:")
print(pd.crosstab(validation['anomaly'], validation['has_fraud'],
                  rownames=['Predicted'], colnames=['Actual Fraud']))

# Kitne actual fraud customers pakde
caught = validation[(validation['anomaly'] == -1) & (validation['has_fraud'] == 1)]
total_fraud_customers = validation['has_fraud'].sum()
print(f"\nTotal fraud customers    : {total_fraud_customers}")
print(f"Caught by Isolation Forest: {len(caught)}")
print(f"Detection rate           : {round(len(caught)/total_fraud_customers*100, 1)}%")

In [ ]:
iso_forest3 = IsolationForest(n_estimators=200,
                               contamination=0.5,
                               random_state=42)

anomaly_scores3 = iso_forest3.fit_predict(features_scaled)
anomaly_score_values3 = iso_forest3.decision_function(features_scaled)

customer_features['anomaly3'] = anomaly_scores3

validation4 = customer_features[['client_id','anomaly3']].merge(
    fraud_per_customer[['client_id','is_high_fraud']],
    on='client_id', how='left'
)

caught4 = validation4[(validation4['anomaly3']==-1) & (validation4['is_high_fraud']==1)]
total_high_fraud = validation4['is_high_fraud'].sum()

print(pd.crosstab(validation4['anomaly3'], validation4['is_high_fraud'],
                  rownames=['Predicted'], colnames=['Actual Fraud']))
print(f"\nDetection rate: {round(len(caught4)/total_high_fraud*100,1)}%")

In [ ]:
umap_df['anomaly'] = customer_features['anomaly3'].values

plt.figure(figsize=(12, 8))

# Normal customers
mask_normal = umap_df['anomaly'] == 1
plt.scatter(umap_df.loc[mask_normal, 'UMAP1'],
            umap_df.loc[mask_normal, 'UMAP2'],
            c='steelblue', label='Normal', alpha=0.5, s=15)

# Anomaly customers
mask_anomaly = umap_df['anomaly'] == -1
plt.scatter(umap_df.loc[mask_anomaly, 'UMAP1'],
            umap_df.loc[mask_anomaly, 'UMAP2'],
            c='red', label='Fraud Risk', alpha=0.9, s=40, marker='X')

plt.title('UMAP — Normal vs Fraud Risk Customers')
plt.xlabel('UMAP 1')
plt.ylabel('UMAP 2')
plt.legend()
plt.tight_layout()
plt.show()